# 07. 종합문화취약지수 설계

- 목적: 격자 단위 문화 접근성 취약도를 종합 지수로 산출함.
- 산출물: 선호반영 H3SFCA 기반 지수와 선호미반영 SFCA 기반 지수를 별도로 생성함.
- 기준: 시설접근성 0.4, 문화다양성 0.2, 노인편의서비스 0.2, 장애인친화시설 0.2
- 장애인·노인 편의 접근성은 `04_1`의 도보+대중교통 통합접근성 결과를 사용함.

## 1. 분석 경로 설정

- 기존 access 산출물을 입력으로 사용함.
- 최종 지수 산출물은 `OUTPUT/final_vulnerability_index`에 저장함.
- 원자료 복제 파일은 생성하지 않음.


In [ ]:
import pathlib
import numpy as np
import pandas as pd

BASE_PATH = pathlib.Path().resolve()

if BASE_PATH.name == "access":
    PROJECT_PATH = BASE_PATH.parents[1]
elif BASE_PATH.name == "notebooks":
    PROJECT_PATH = BASE_PATH.parent
elif BASE_PATH.name != "oracle_mnc_project" and (BASE_PATH / "oracle_mnc_project").exists():
    PROJECT_PATH = BASE_PATH / "oracle_mnc_project"
else:
    PROJECT_PATH = BASE_PATH

ACCESS_OUTPUT_PATH = PROJECT_PATH / "notebooks" / "access" / "OUTPUT"
H3_PATH = ACCESS_OUTPUT_PATH / "h3sfca"
DIVERSITY_PATH = ACCESS_OUTPUT_PATH / "diversity"
DISABILITY_ELDERLY_PATH = ACCESS_OUTPUT_PATH / "disability_elderly_accessibility_mode_consistent"
FINAL_OUTPUT_PATH = ACCESS_OUTPUT_PATH / "final_vulnerability_index"
DOCS_PATH = PROJECT_PATH / "notebooks" / "access" / "docs"

FINAL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
DOCS_PATH.mkdir(parents=True, exist_ok=True)

PREFERENCE_ACCESS_PATH = H3_PATH / "h3sfca_격자_중분류_접근성.csv"
NO_PREFERENCE_ACCESS_PATH = H3_PATH / "sfca_no_preference_격자_중분류_접근성.csv"
DIVERSITY_TABLE_PATH = DIVERSITY_PATH / "grid_pop_access_diversity.csv"
DISABILITY_ELDERLY_TABLE_PATH = DISABILITY_ELDERLY_PATH / "장애인_노인_중분류이동기준_E2SFCA_격자.csv"

print("PROJECT_PATH:", PROJECT_PATH)
print("PREFERENCE_ACCESS_PATH:", PREFERENCE_ACCESS_PATH.exists(), PREFERENCE_ACCESS_PATH)
print("NO_PREFERENCE_ACCESS_PATH:", NO_PREFERENCE_ACCESS_PATH.exists(), NO_PREFERENCE_ACCESS_PATH)
print("DIVERSITY_TABLE_PATH:", DIVERSITY_TABLE_PATH.exists(), DIVERSITY_TABLE_PATH)
print("DISABILITY_ELDERLY_TABLE_PATH:", DISABILITY_ELDERLY_TABLE_PATH.exists(), DISABILITY_ELDERLY_TABLE_PATH)
print("FINAL_OUTPUT_PATH:", FINAL_OUTPUT_PATH)


## 2. 지표 수식

- 접근성이 높을수록 양호하므로 취약점수는 z-score의 부호를 반전함.
- 중분류별 시설 접근성은 중분류 안에서 표준화한 뒤 격자 단위로 합산함.
- 최종 결합 전 각 하위지표를 다시 z-score 표준화함.

\[
V_{i,c}^{access} = -Z(A_{i,c})
\]

\[
V_i^{access} = \sum_c V_{i,c}^{access}
\]

\[
V_i^{final} = 0.4Z(V_i^{access}) + 0.2Z(V_i^{diversity}) + 0.2V_i^{elderly} + 0.2V_i^{disabled}
\]

- 값이 클수록 취약함.
- 취약지역 판정은 문화누리대상자 추정인구가 있는 격자만 대상으로 함.


## 3. 공통 함수 정의

- z-score는 문화누리대상자 추정인구가 있는 격자를 기준으로 계산함.
- 표준편차가 0인 경우 해당 지표의 변별력이 없으므로 0으로 처리함.
- 최종 취약 백분위는 분석대상 격자 안에서 계산함.


In [ ]:
def zscore_by_mask(series, mask):
    result = pd.Series(np.nan, index=series.index, dtype="float64")
    values = pd.to_numeric(series, errors="coerce")
    mean_value = values.loc[mask].mean()
    std_value = values.loc[mask].std(ddof=0)
    
    if pd.isna(std_value) or np.isclose(std_value, 0):
        result.loc[mask] = 0
    else:
        result.loc[mask] = (values.loc[mask] - mean_value) / std_value
    
    return result


def add_percentile_and_grade(result):
    mask = result["분석대상"]
    result["최종취약지수_백분위"] = np.nan
    result.loc[mask, "최종취약지수_백분위"] = (
        result.loc[mask, "최종취약지수_z"].rank(method="average", pct=True) * 100
    )
    
    result["취약등급"] = "분석제외"
    result.loc[mask & (result["최종취약지수_백분위"] >= 90), "취약등급"] = "매우취약"
    result.loc[mask & (result["최종취약지수_백분위"].between(80, 90, inclusive="left")), "취약등급"] = "취약"
    result.loc[mask & (result["최종취약지수_백분위"].between(20, 80, inclusive="left")), "취약등급"] = "보통"
    result.loc[mask & (result["최종취약지수_백분위"] < 20), "취약등급"] = "양호"
    
    return result


def add_main_causes(result):
    cause_cols = {
        "시설접근성": "시설접근성취약점수_z",
        "문화다양성": "다양성취약점수_z",
        "노인편의서비스": "노인편의취약점수_z",
        "장애인친화시설": "장애인친화취약점수_z",
    }
    cause_matrix = result[list(cause_cols.values())].copy()
    cause_matrix.columns = list(cause_cols.keys())
    
    result["주요취약원인1"] = ""
    result["주요취약원인2"] = ""
    mask = result["분석대상"]
    
    ordered = np.argsort(-cause_matrix.loc[mask].to_numpy(), axis=1)
    names = np.array(cause_matrix.columns)
    result.loc[mask, "주요취약원인1"] = names[ordered[:, 0]]
    result.loc[mask, "주요취약원인2"] = names[ordered[:, 1]]
    
    return result


## 4. 공통 보조지표 불러오기

- 문화다양성은 접근 가능한 중분류 비율의 결핍값을 사용함.
- 장애인·노인 편의 접근성은 `04_1`에서 산출한 도보+대중교통 통합접근성을 사용함.
- 장애인·노인 도달 가능 중분류와 부족 중분류는 대시보드 원인 설명용으로 보존함.
- 결측은 격자 병합 후 수치형 0, 문자형 공백으로 처리함.

In [ ]:
diversity = pd.read_csv(DIVERSITY_TABLE_PATH, encoding="utf-8-sig")
disability_elderly = pd.read_csv(DISABILITY_ELDERLY_TABLE_PATH, encoding="utf-8-sig")

base_cols = [
    "GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y",
    "추정_인구수", "문화누리대상자_추정_인구수"
]

diversity = diversity[[
    "GRID_CD", "diversity_type_n", "reachable_store_n",
    "diversity_max_type_n", "diversity_ratio", "diversity_vulnerability"
]].copy()

disability_elderly_cols = [
    "GRID_CD", "장애인_수요인구수", "노령인구_수요인구수",
    "장애인친화시설_도보_E2SFCA", "장애인친화시설_대중교통_E2SFCA",
    "장애인친화시설_도보접근성점수", "장애인친화시설_대중교통접근성점수",
    "장애인친화시설_통합접근성", "장애인친화시설_통합도달가맹점수",
    "장애인친화시설_도달가능중분류", "장애인친화시설_부족중분류",
    "노인편의서비스_도보_E2SFCA", "노인편의서비스_대중교통_E2SFCA",
    "노인편의서비스_도보접근성점수", "노인편의서비스_대중교통접근성점수",
    "노인편의서비스_통합접근성", "노인편의서비스_통합도달가맹점수",
    "노인편의서비스_도달가능중분류", "노인편의서비스_부족중분류",
]
missing_disability_cols = [col for col in disability_elderly_cols if col not in disability_elderly.columns]
if missing_disability_cols:
    raise KeyError(f"장애인·노인 04_1 입력 컬럼이 없습니다: {missing_disability_cols}")

disability_elderly = disability_elderly[disability_elderly_cols].copy()

for col in diversity.columns:
    if col != "GRID_CD":
        diversity[col] = pd.to_numeric(diversity[col], errors="coerce").fillna(0)

text_cols = [
    "장애인친화시설_도달가능중분류", "장애인친화시설_부족중분류",
    "노인편의서비스_도달가능중분류", "노인편의서비스_부족중분류",
]
for col in disability_elderly.columns:
    if col == "GRID_CD":
        continue
    if col in text_cols:
        disability_elderly[col] = disability_elderly[col].fillna("").astype(str)
    else:
        disability_elderly[col] = pd.to_numeric(disability_elderly[col], errors="coerce").fillna(0)

print("다양성 테이블 구조:", diversity.shape)
print("장애인/노인 04_1 접근성 테이블 구조:", disability_elderly.shape)
print("다양성 GRID 중복:", diversity["GRID_CD"].duplicated().sum())
print("장애인/노인 GRID 중복:", disability_elderly["GRID_CD"].duplicated().sum())

## 5. 시설 접근성 취약점수 생성 함수

- 중분류별 접근성지수는 같은 중분류 안에서 z-score 표준화함.
- 접근성은 높을수록 양호하므로 부호를 반전해 취약점수로 변환함.
- 격자별 시설 접근성 취약점수는 중분류별 취약 z-score를 합산함.
- 최종 결합에는 다시 z-score 표준화한 시설접근성취약점수_z를 사용함.


In [ ]:
def build_access_component(access_path, model_label):
    usecols = [
        "GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y",
        "추정_인구수", "문화누리대상자_추정_인구수",
        "중분류", "접근성지수", "접근가능_가맹점수"
    ]
    access = pd.read_csv(access_path, encoding="utf-8-sig", usecols=usecols)
    
    access["접근성지수"] = pd.to_numeric(access["접근성지수"], errors="coerce").fillna(0)
    access["접근가능_가맹점수"] = pd.to_numeric(access["접근가능_가맹점수"], errors="coerce").fillna(0)
    access["문화누리대상자_추정_인구수"] = pd.to_numeric(access["문화누리대상자_추정_인구수"], errors="coerce").fillna(0)
    access["추정_인구수"] = pd.to_numeric(access["추정_인구수"], errors="coerce").fillna(0)
    access["분석대상"] = access["문화누리대상자_추정_인구수"] > 0
    
    access["중분류별_접근성_z"] = np.nan
    access["중분류별_시설접근성취약_z"] = np.nan
    
    for category in sorted(access["중분류"].dropna().unique()):
        idx = access["중분류"].eq(category)
        target_idx = idx & access["분석대상"]
        z = zscore_by_mask(access.loc[idx, "접근성지수"], access.loc[idx, "분석대상"])
        access.loc[idx, "중분류별_접근성_z"] = z
        access.loc[idx, "중분류별_시설접근성취약_z"] = -z
        print(f"{model_label} | {category}: 분석대상 {int(target_idx.sum()):,}, 평균접근성 {access.loc[target_idx, '접근성지수'].mean():.6f}")
    
    group_cols = base_cols
    access_component = (
        access
        .groupby(group_cols, as_index=False)
        .agg(
            시설접근성취약점수_raw=("중분류별_시설접근성취약_z", "sum"),
            시설접근성취약점수_mean=("중분류별_시설접근성취약_z", "mean"),
            접근가능중분류수=("접근가능_가맹점수", lambda x: int((x > 0).sum())),
            평균접근가능가맹점수=("접근가능_가맹점수", "mean")
        )
    )
    
    target_access = access[access["분석대상"]].copy()
    target_access = target_access.sort_values(
        ["GRID_CD", "중분류별_시설접근성취약_z"],
        ascending=[True, False]
    )
    worst_category = (
        target_access
        .groupby("GRID_CD", as_index=False)
        .first()[["GRID_CD", "중분류", "중분류별_시설접근성취약_z"]]
        .rename(columns={
            "중분류": "시설접근성_최취약중분류",
            "중분류별_시설접근성취약_z": "시설접근성_최취약중분류_z"
        })
    )
    
    access_component = access_component.merge(worst_category, on="GRID_CD", how="left")
    access_component["지수모형"] = model_label
    
    print()
    print(f"{model_label} 시설 접근성 컴포넌트 구조:", access_component.shape)
    print("격자 수:", access_component["GRID_CD"].nunique())
    print("접근성 중분류 수:", access["중분류"].nunique())
    print("분석대상 격자 수:", access_component["문화누리대상자_추정_인구수"].gt(0).sum())
    
    return access_component


## 6. 종합취약지수 생성 함수

- 시설접근성, 다양성, 노인편의, 장애인친화 지표를 격자 단위로 병합함.
- 각 하위지표는 분석대상 격자 기준 z-score로 표준화함.
- 장애인·노인 편의 지표는 04_1의 통합접근성을 사용하고, 접근성이 높을수록 양호하므로 부호를 반전함.
- 최종 취약지수는 `0.4/0.2/0.2/0.2` 가중합으로 계산함.

In [ ]:
def build_final_index(access_component, model_label):
    result = access_component.merge(diversity, on="GRID_CD", how="left")
    result = result.merge(disability_elderly, on="GRID_CD", how="left")
    
    numeric_fill_cols = [
        "diversity_type_n", "reachable_store_n", "diversity_max_type_n",
        "diversity_ratio", "diversity_vulnerability",
        "장애인_수요인구수", "노령인구_수요인구수",
        "장애인친화시설_도보_E2SFCA", "장애인친화시설_대중교통_E2SFCA",
        "장애인친화시설_도보접근성점수", "장애인친화시설_대중교통접근성점수",
        "장애인친화시설_통합접근성", "장애인친화시설_통합도달가맹점수",
        "노인편의서비스_도보_E2SFCA", "노인편의서비스_대중교통_E2SFCA",
        "노인편의서비스_도보접근성점수", "노인편의서비스_대중교통접근성점수",
        "노인편의서비스_통합접근성", "노인편의서비스_통합도달가맹점수",
    ]
    text_fill_cols = [
        "장애인친화시설_도달가능중분류", "장애인친화시설_부족중분류",
        "노인편의서비스_도달가능중분류", "노인편의서비스_부족중분류",
    ]
    result[numeric_fill_cols] = result[numeric_fill_cols].fillna(0)
    result[text_fill_cols] = result[text_fill_cols].fillna("")
    result["분석대상"] = result["문화누리대상자_추정_인구수"] > 0
    mask = result["분석대상"]
    
    result["시설접근성취약점수_z"] = zscore_by_mask(result["시설접근성취약점수_raw"], mask)
    result["다양성취약점수_z"] = zscore_by_mask(result["diversity_vulnerability"], mask)
    result["노인편의취약점수_z"] = -zscore_by_mask(result["노인편의서비스_통합접근성"], mask)
    result["장애인친화취약점수_z"] = -zscore_by_mask(result["장애인친화시설_통합접근성"], mask)
    
    result["최종취약지수_z"] = (
        0.4 * result["시설접근성취약점수_z"]
        + 0.2 * result["다양성취약점수_z"]
        + 0.2 * result["노인편의취약점수_z"]
        + 0.2 * result["장애인친화취약점수_z"]
    )
    result.loc[~mask, "최종취약지수_z"] = np.nan
    
    result = add_percentile_and_grade(result)
    result = add_main_causes(result)
    
    ordered_cols = [
        "GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y",
        "추정_인구수", "문화누리대상자_추정_인구수", "분석대상", "지수모형",
        "최종취약지수_z", "최종취약지수_백분위", "취약등급",
        "시설접근성취약점수_z", "다양성취약점수_z", "노인편의취약점수_z", "장애인친화취약점수_z",
        "시설접근성취약점수_raw", "시설접근성취약점수_mean", "시설접근성_최취약중분류", "시설접근성_최취약중분류_z",
        "diversity_type_n", "reachable_store_n", "diversity_ratio", "diversity_vulnerability",
        "장애인_수요인구수", "장애인친화시설_도보_E2SFCA", "장애인친화시설_대중교통_E2SFCA",
        "장애인친화시설_도보접근성점수", "장애인친화시설_대중교통접근성점수",
        "장애인친화시설_통합접근성", "장애인친화시설_통합도달가맹점수",
        "장애인친화시설_도달가능중분류", "장애인친화시설_부족중분류",
        "노령인구_수요인구수", "노인편의서비스_도보_E2SFCA", "노인편의서비스_대중교통_E2SFCA",
        "노인편의서비스_도보접근성점수", "노인편의서비스_대중교통접근성점수",
        "노인편의서비스_통합접근성", "노인편의서비스_통합도달가맹점수",
        "노인편의서비스_도달가능중분류", "노인편의서비스_부족중분류",
        "주요취약원인1", "주요취약원인2", "접근가능중분류수", "평균접근가능가맹점수"
    ]
    result = result[ordered_cols].copy()
    
    print()
    print(f"{model_label} 최종취약지수 구조:", result.shape)
    print("분석대상 격자 수:", int(result["분석대상"].sum()))
    print("취약등급 분포")
    print(result["취약등급"].value_counts(dropna=False).to_string())
    print()
    print("최종취약지수 요약")
    print(result.loc[mask, ["최종취약지수_z", "최종취약지수_백분위"]].describe().to_string())
    
    return result

## 7. 선호반영 H3SFCA 기반 종합문화취약지수

- 시설 접근성 하위지표는 H3SFCA 기본 시나리오를 사용함.
- 기본 시나리오: 구간형 거리감쇠 + 문화누리대상자 수요가중치 1.2
- 음악·체육용품은 기존 합의대로 선호 미반영 SFCA 대체값이 반영된 결과를 사용함.


In [ ]:
preference_access_component = build_access_component(
    PREFERENCE_ACCESS_PATH,
    "선호반영_H3SFCA"
)

preference_final = build_final_index(
    preference_access_component,
    "선호반영_H3SFCA"
)


## 8. 선호미반영 SFCA 기반 종합문화취약지수

- 시설 접근성 하위지표는 선호확률을 반영하지 않은 SFCA 결과를 사용함.
- 총 추정인구를 기준수요로 사용한 접근성 결과임.
- 선호반영 지수와 별도 산출물로 저장함.


In [ ]:
no_preference_access_component = build_access_component(
    NO_PREFERENCE_ACCESS_PATH,
    "선호미반영_SFCA"
)

no_preference_final = build_final_index(
    no_preference_access_component,
    "선호미반영_SFCA"
)


## 9. 산출물 저장 및 시군구 요약

- 선호반영 지수와 선호미반영 지수를 별도 CSV로 저장함.
- 시군구 요약은 결과 검토용으로 함께 저장함.
- 취약지역 판정은 분석대상 격자 안에서 상위 20%를 기준으로 함.


In [ ]:
preference_output_path = FINAL_OUTPUT_PATH / "종합문화취약지수_선호반영_H3SFCA.csv"
no_preference_output_path = FINAL_OUTPUT_PATH / "종합문화취약지수_선호미반영_SFCA.csv"
summary_output_path = FINAL_OUTPUT_PATH / "종합문화취약지수_시군구요약.csv"

preference_final.to_csv(preference_output_path, index=False, encoding="utf-8-sig")
no_preference_final.to_csv(no_preference_output_path, index=False, encoding="utf-8-sig")

combined = pd.concat([preference_final, no_preference_final], ignore_index=True)
summary = (
    combined[combined["분석대상"]]
    .groupby(["지수모형", "시군구"], as_index=False)
    .agg(
        분석격자수=("GRID_CD", "nunique"),
        문화누리대상자_추정인구=("문화누리대상자_추정_인구수", "sum"),
        평균최종취약지수_z=("최종취약지수_z", "mean"),
        중앙최종취약지수_z=("최종취약지수_z", "median"),
        평균취약백분위=("최종취약지수_백분위", "mean"),
        매우취약격자수=("취약등급", lambda x: int((x == "매우취약").sum())),
        취약격자수=("취약등급", lambda x: int(x.isin(["매우취약", "취약"]).sum()))
    )
)
summary["취약격자비율"] = summary["취약격자수"] / summary["분석격자수"]
summary.to_csv(summary_output_path, index=False, encoding="utf-8-sig")

print("저장 완료")
print("선호반영:", preference_output_path)
print("선호미반영:", no_preference_output_path)
print("시군구요약:", summary_output_path)

print()
print("선호반영 H3SFCA 평균취약지수 상위 10개 구")
print(
    summary[summary["지수모형"].eq("선호반영_H3SFCA")]
    .sort_values("평균최종취약지수_z", ascending=False)
    .head(10)
    [["시군구", "평균최종취약지수_z", "평균취약백분위", "취약격자수", "취약격자비율"]]
    .to_string(index=False)
)

print()
print("선호미반영 SFCA 평균취약지수 상위 10개 구")
print(
    summary[summary["지수모형"].eq("선호미반영_SFCA")]
    .sort_values("평균최종취약지수_z", ascending=False)
    .head(10)
    [["시군구", "평균최종취약지수_z", "평균취약백분위", "취약격자수", "취약격자비율"]]
    .to_string(index=False)
)


## 10. 문서 메모 저장

- 사용 데이터, 전처리 방식, 주요 산출물을 docs 메모장에 기록함.
- 선호미반영 종합문화취약지수는 별도 산출물로 분리함.


In [ ]:
doc_text = """# 종합문화취약지수

## 사용 데이터
- 선호반영 시설 접근성: h3sfca_격자_중분류_접근성.csv
- 선호미반영 시설 접근성: sfca_no_preference_격자_중분류_접근성.csv
- 문화시설 다양성: grid_pop_access_diversity.csv
- 장애인/노인 편의 접근성: 장애인_노인_중분류이동기준_E2SFCA_격자.csv

## 전처리/분석 방식
- 분석 단위는 서울 100m 격자임.
- 문화누리대상자 추정인구가 0보다 큰 격자를 최종 취약지수 판정 대상으로 설정함.
- 중분류별 시설 접근성은 중분류 내부에서 z-score 표준화 후 부호를 반전해 취약점수로 변환함.
- 격자별 시설 접근성 취약점수는 중분류별 취약 z-score를 합산해 산출함.
- 문화다양성은 diversity_vulnerability를 사용하고 z-score 표준화함.
- 노인편의서비스와 장애인친화시설 접근성은 04_1의 도보+대중교통 통합접근성을 사용하고, z-score 표준화 후 부호를 반전함.
- 도달 가능 중분류와 부족 중분류는 대시보드 원인 설명용으로 보존함.
- 최종 지수는 시설접근성 0.4, 문화다양성 0.2, 노인편의서비스 0.2, 장애인친화시설 0.2로 가중합함.
- 선호반영 H3SFCA 기반 지수와 선호미반영 SFCA 기반 지수를 별도 파일로 저장함.

## 주요 산출물
- 종합문화취약지수_선호반영_H3SFCA.csv
- 종합문화취약지수_선호미반영_SFCA.csv
- 종합문화취약지수_시군구요약.csv
"""

doc_path = DOCS_PATH / "final_vulnerability_index_전처리_사용데이터.txt"
doc_path.write_text(doc_text, encoding="utf-8")
print("문서 메모 저장:", doc_path)